In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/movie-recommendation-hackathon-2026/sample_submission.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/imdb_data.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv
/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv


In [3]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [4]:
train_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/train.csv')

train_df['datetime'] = pd.to_datetime(train_df['timestamp'], unit='s')

train_df.head()

,userId,movieId,rating,timestamp,datetime
0,5163,57669,4.0,1518349992,2018-02-11 11:53:12
1,106343,5,4.5,1206238739,2008-03-23 02:18:59
2,146790,5459,5.0,1076215539,2004-02-08 04:45:39
3,106362,32296,2.0,1423042565,2015-02-04 09:36:05
4,9041,366,3.0,833375837,1996-05-29 13:17:17


In [5]:
print("Earliest date: ", train_df['datetime'].min())
print("Latest date: ", train_df['datetime'].max())

Earliest date:  1995-01-09 11:46:49
Latest date:  2019-11-21 09:12:29


In [6]:
tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/tags.csv')

tags_df['datetime'] = pd.to_datetime(tags_df['timestamp'], unit='s')

tags_df.head()

,userId,movieId,tag,timestamp,datetime
0,3,260,classic,1439472355,2015-08-13 13:25:55
1,3,260,sci-fi,1439472256,2015-08-13 13:24:16
2,4,1732,dark comedy,1573943598,2019-11-16 22:33:18
3,4,1732,great dialogue,1573943604,2019-11-16 22:33:24
4,4,7569,so bad it's good,1573943455,2019-11-16 22:30:55


In [7]:
movies_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/movies.csv')

movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
links_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/links.csv')

links_df.head()

,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0
3,4,114885,31357.0
4,5,113041,11862.0


In [9]:
genome_tags_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_tags.csv')

genome_tags_df.head()

,tagId,tag
0,1,007
1,2,007 (series)
2,3,18th century
3,4,1920s
4,5,1930s


In [10]:
genome_scores_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/genome_scores.csv')

genome_scores_df.head()

,movieId,tagId,relevance
0,1,1,0.02875
1,1,2,0.02375
2,1,3,0.06250
3,1,4,0.07575
4,1,5,0.14075


## More EDA on my Part

In [11]:
# train_df['userId'].sort_values().unique().tolist()
grouped_data_users = train_df.groupby('userId')

grouped_data_users.get_group(106343)['rating'].sort_values(ascending=True).unique()

# You get all the movies a user has rated after grouping

array([0.5, 1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5, 5. ])

In [12]:
tags_df['userId'].values

array([     3,      3,      4, ..., 162534, 162534, 162534])

In [13]:
grouped_data_tags = tags_df.groupby('userId')

grouped_data_tags.get_group(4)

# You get all the tags applied by a user to movies after grouping

,userId,movieId,tag,timestamp,datetime
2,4,1732,dark comedy,1573943598,2019-11-16 22:33:18
3,4,1732,great dialogue,1573943604,2019-11-16 22:33:24
4,4,7569,so bad it's good,1573943455,2019-11-16 22:30:55
5,4,44665,unreliable narrators,1573943619,2019-11-16 22:33:39
6,4,115569,tense,1573943077,2019-11-16 22:24:37
7,4,115713,artificial intelligence,1573942979,2019-11-16 22:22:59
8,4,115713,philosophical,1573943033,2019-11-16 22:23:53
9,4,115713,tense,1573943042,2019-11-16 22:24:02
10,4,148426,so bad it's good,1573942965,2019-11-16 22:22:45
11,4,164909,cliche,1573943721,2019-11-16 22:35:21


In [14]:
grouped_data_movies = movies_df.groupby('movieId')

grouped_data_movies.get_group(260)

,movieId,title,genres
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi


In [15]:
train_df['movieId'].sort_values().reset_index(drop=True).equals(movies_df['movieId'].sort_values().reset_index(drop=True))

False

In [16]:
movies_df[movies_df['movieId'] == 57669]

,movieId,title,genres
12111,57669,In Bruges (2008),Comedy|Crime|Drama|Thriller


## Function prototyping

In [17]:
def cbf_recommender_fn(train_df, movie_df, user_id, rating_threshold, n=10):
    """
    This function implements a recommender system for movies by
    analyzing what the movies the user likes and recommending
    movies similar to the one the user likes

    Parameters:
        train_df - Dataframe with rows representing a rating for a movie by one user (userId, movieId, rating, timestamp)
        movie_df - Dataframe containing details for a movie (movieId, title, genre)
        user_id - The id of the user to use in the recommender system
        rating_threshold - The threshold with which to filter movies that have poor ratings
        n - number of recommendations to return to the user

    Returns:
        - A dataframe with the top n movie recommendations
    """
    
    '''Step one: Check if userId and rating_threshold is valid'''
    if user_id not in train_df['userId'].values:
        return f"Please Enter a valid user Id"

    valid_ratings = np.arange(0.5, 5.5, 0.5)
    if rating_threshold not in valid_ratings:
        return ("""
            Your rating doesn't meet the required threshold.
            Input a number between 0.5 and 5.0 and ensure that
            It ends with .5 or .0
            1.0, 1.5, 2.0, 2.5 are valid but 1.8, 2.6 are not
            """
        )
    
    '''Step two: Group the train_df by userId'''
    user_data = train_df.groupby('userId').get_group(user_id)

    '''Step three: Use the rating_threshold to filter poorly rated movies'''
    filtered_user_data = user_data[user_data['rating'] >= rating_threshold]

    '''Step Four: Reference movies in the movies_df after applying the filter'''
    movie_ids = filtered_user_data['movieId'].values

    if not pd.Series(movie_ids).isin(movie_df['movieId']).all():
        return "Please ensure the Movie Id is valid"
    
    movie_details = movie_df[movie_df['movieId'].isin(movie_ids)]

    '''Step Five: Represent the movie_df genres column numerically'''
    genre_list = movie_df['genres'].str.split('|')

    mlb = MultiLabelBinarizer()
    genre_matrix = mlb.fit_transform(genre_list)

    # Find the positions of the user's liked movies
    liked_positions = np.flatnonzero(
        movie_df['movieId'].isin(movie_ids).to_numpy()
    )

    # Extract only the liked movies' feature vectors
    liked_genres = genre_matrix[liked_positions]

    liked_genres_df = pd.DataFrame(
        liked_genres,
        columns=mlb.classes_,
        index=movie_df.iloc[liked_positions]['title']
    )

    genre_df = pd.DataFrame(
        genre_matrix,
        columns=mlb.classes_,
        index=movie_df['title']
    )

    # Compare all movies against the liked movies
    cosine_sim = cosine_similarity(genre_matrix, liked_genres)

    # Calculate an average similairty score for each movie
    avg_sim = cosine_sim.mean(axis=1)

    # avg_sim_df = pd.DataFrame(
    #     avg_sim,
    #     index=movie_df['title'],
    #     columns=['avg_sim']
    # )
    avg_sim_df = pd.DataFrame({
        'movieId': movie_df['movieId'].values,
        'title': movie_df['title'].values,
        'avg_similarity': avg_sim
    })

    '''Step Six: Remove movies the user has already rated'''
    rated_movies_ids = train_df.loc[
        train_df['userId'] == user_id, 'movieId'
    ].unique()
    
    recommendations = avg_sim_df[
        ~avg_sim_df['movieId'].isin(rated_movies_ids)
    ]

    recommendations = recommendations.sort_values(
        by='avg_similarity',
        ascending=False
    )

    recommendations['similarity (%)'] = (
        recommendations['avg_similarity'] * 100
    ).round(2).astype(int)
    
    return recommendations[['title', 'similarity (%)']].head(n).reset_index(drop=True)

In [18]:
cbf_recommender_fn(train_df, movies_df, 106343, 5.0, 12)

,title,similarity (%)
0,Keeping the Faith (2000),56
1,Bossa Nova (2000),56
2,Limelight (1952),56
3,Drinking Buddies (2013),56
4,Seven Something (2012),56
5,How to Date an Otaku Girl (2009),56
6,No Longer Heroine (2015),56
7,Down Argentine Way (1940),56
8,In Lieu of Flowers (2013),56
9,Love in a Puff (2010),56


## Testing the function

In [19]:
test_df = pd.read_csv('/kaggle/input/competitions/movie-recommendation-hackathon-2026/test.csv')

test_df.head()

,userId,movieId
0,1,2011
1,1,4144
2,1,5767
3,1,6711
4,1,7318


In [20]:
test_df.shape

(5000019, 2)

In [21]:
subset = test_df['movieId'].head()

mask = test_df['movieId'].isin(train_df['movieId'])

mask.value_counts()

movieId
True     4993636
False       6383
Name: count, dtype: int64

In [22]:
train_df.shape

(10000038, 5)

In [23]:
test_df.columns

Index(['userId', 'movieId'], dtype='object')

In [24]:
train_df.columns

Index(['userId', 'movieId', 'rating', 'timestamp', 'datetime'], dtype='object')

## Function Prototyping V2 for Content-based Filtering

In [43]:
cutoff_date = pd.to_datetime('2014-01-01')

train_data = train_df[train_df['datetime'] < cutoff_date]
validation_data = train_df[train_df['datetime'] >= cutoff_date]

In [46]:
mlb = MultiLabelBinarizer()

genres_encoded = mlb.fit_transform(
    movies_df['genres'].str.split('|')
)

genres_df = pd.DataFrame(
    genres_encoded,
    columns=mlb.classes_,
    index=movies_df['title']
)

genres_df.head()

,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
title,,,,,,,,,,,,,,,,,,,,
Toy Story (1995),0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
Jumanji (1995),0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
Grumpier Old Men (1995),0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
Waiting to Exhale (1995),0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [49]:
ratings_with_genres = train_data.merge(
    movies_df[['movieId', 'genres']],
    on='movieId',
    how='left'
)

ratings_with_genres['genres'] = (
    ratings_with_genres['genres'].str.split('|')
)

ratings_with_genres = ratings_with_genres.explode('genres')

ratings_with_genres

,userId,movieId,rating,timestamp,datetime,genres
0,106343,5,4.5,1206238739,2008-03-23 02:18:59,Comedy
1,146790,5459,5.0,1076215539,2004-02-08 04:45:39,Action
1,146790,5459,5.0,1076215539,2004-02-08 04:45:39,Comedy
1,146790,5459,5.0,1076215539,2004-02-08 04:45:39,Sci-Fi
2,9041,366,3.0,833375837,1996-05-29 13:17:17,Drama
...,...,...,...,...,...,...
6782203,154807,56782,4.0,1227674807,2008-11-26 04:46:47,Drama
6782203,154807,56782,4.0,1227674807,2008-11-26 04:46:47,Western
6782204,139457,1009,4.0,858984862,1997-03-21 22:54:22,Adventure
6782204,139457,1009,4.0,858984862,1997-03-21 22:54:22,Children


In [53]:
user_genre_preferences = (
    ratings_with_genres.groupby(['userId', 'genres'])['rating']
    .mean()
    .unstack()
)

user_genre_preferences

genres,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
userId,,,,,,,,,,,,,,,,,,,,
1,NaN,5.000000,4.666667,NaN,NaN,3.791667,4.125000,NaN,3.916667,3.500000,NaN,NaN,NaN,3.25,3.5,3.833333,4.500000,5.000000,3.900000,NaN
2,NaN,3.448276,3.846154,2.875,3.562500,3.083333,2.388889,NaN,3.243243,3.676471,NaN,NaN,4.50,2.75,3.0,3.031250,3.857143,3.772727,3.642857,3.000000
5,NaN,3.142857,4.000000,4.000,3.666667,3.750000,4.250000,NaN,3.875000,3.750000,NaN,5.000000,5.00,4.00,4.6,3.833333,4.000000,3.818182,4.500000,3.000000
6,NaN,4.000000,4.000000,NaN,4.000000,4.000000,5.000000,NaN,4.250000,NaN,NaN,NaN,NaN,NaN,NaN,4.000000,4.333333,NaN,NaN,NaN
7,NaN,3.000000,4.000000,4.000,3.666667,4.000000,3.000000,NaN,4.600000,3.500000,NaN,NaN,4.00,4.00,NaN,4.500000,NaN,3.000000,5.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162535,NaN,4.000000,4.000000,NaN,2.500000,3.714286,4.000000,NaN,4.090909,2.666667,NaN,NaN,NaN,NaN,4.5,4.142857,2.500000,4.333333,5.000000,NaN
162537,NaN,4.000000,4.400000,4.000,3.800000,4.208333,NaN,3.0,3.800000,3.000000,NaN,4.000000,NaN,4.40,NaN,4.071429,4.500000,4.000000,5.000000,NaN
162539,NaN,4.200000,4.250000,NaN,NaN,4.666667,5.000000,NaN,4.818182,NaN,5.0,5.000000,NaN,NaN,5.0,4.571429,4.250000,4.500000,4.750000,NaN
